In [3]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# import necessary libraries
import pandas as pd
import gensim 
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string
import nltk
import re


In [6]:
df = pd.read_csv("https://github.com/Ankit152/IMDB-sentiment-analysis/raw/refs/heads/master/IMDB-Dataset.csv")

In [7]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [8]:
from sklearn.model_selection import train_test_split

In [9]:
X_train,X_test,Y_train,Y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.5,
    random_state=42,
    stratify=df['sentiment'] # to make sure balance of split in data 12.5 positive and 12.5 negative
)

In [10]:
print('Training set:',X_train.shape)
print(Y_train.value_counts())

print("\nTesting set:",X_test.shape)
print(Y_test.value_counts())

Training set: (25000,)
sentiment
positive    12500
negative    12500
Name: count, dtype: int64

Testing set: (25000,)
sentiment
negative    12500
positive    12500
Name: count, dtype: int64


In [11]:
nltk.download('punkt') # needed for tokenozation
nltk.download('stopwords') #    needed for removation of stopwords 

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [12]:
nltk.download('punkt_tab') 
nltk.download('stopwords')

# preprocess function 
def preprocess(text):
    text = re.sub(r'<[^>]+>', ' ', text)  # Remove HTML tags
    # lowercase , remove punctuation and tokenize
    text = text.lower()
    text = ''.join([char for char in text if char not in string.punctuation])
    words = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    return words

# Preprocess training and testing reviews separately
processed_train = X_train.apply(preprocess)
processed_test = X_test.apply(preprocess)


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [13]:
processed_train

3055     [trios, vignettes, insightful, quite, enjoyabl...
47326    [never, heard, one, owner, local, dvd, rental,...
48869    [excellent, hard, find, trippy, world, war, sp...
48623    [huge, fan, big, loud, trashy, completely, stu...
39657    [easily, lucio, fulcis, respected, film, dont,...
                               ...                        
21552    [recent, years, seen, number, biopics, famous,...
10610    [may, say, johnson, elvis, work, possible, see...
20104    [title, refers, questionable, poker, hand, six...
35597    [far, pathetic, movie, indian, cinema, cinema,...
27491    [movie, forever, left, impression, watched, fr...
Name: review, Length: 25000, dtype: object

In [14]:
processed_test

16346    [sixyearold, boy, whos, dinosaurs, love, movie...
42137    [voted, 1, film, noticed, 75, people, voted, 1...
21867    [arent, talking, animals, big, lavish, song, p...
12577    [ever, think, like, watching, horror, movie, g...
25444    [expecting, love, moviefilm, noir, serial, kil...
                               ...                        
32628    [read, comments, site, feel, compelled, write,...
8352     [people, expect, jurassic, park, 3, full, surp...
49159    [fan, probably, enjoy, dont, know, misty, mund...
38453    [ever, since, played, goon, lone, wolf, mcquad...
47518    [found, movie, nearly, impossible, watch, supe...
Name: review, Length: 25000, dtype: object

In [15]:
# train word2vec model on the processed text data 
model = Word2Vec(sentences=processed_train,vector_size=100,window=5,min_count=1,workers=4)

In [16]:
model.wv.vector_size

100

In [17]:
model.wv['wonderful']

array([-2.2472692 , -1.5980722 , -0.8376709 , -2.2255106 , -0.51418054,
       -2.4363782 ,  0.03515312,  1.7936878 , -0.41163227, -0.03606943,
        0.44078773, -1.2876767 , -0.83082485,  1.5269924 , -0.67110866,
        0.37299475,  0.17077774,  0.32319507, -0.7121836 ,  0.17616549,
        2.4556837 ,  0.3944141 ,  0.5770457 ,  0.60857064,  0.24549964,
        2.0697212 , -1.0166891 , -0.08445015, -1.3585631 , -1.2911634 ,
       -0.149945  ,  0.9040549 ,  1.4925935 ,  0.36997098, -0.5305682 ,
        1.6901739 , -1.1102325 , -2.7005136 , -1.5939206 ,  0.22314376,
        0.18740696,  0.18565494, -0.8811088 , -1.3745177 ,  1.0780011 ,
        0.63624084, -0.61105466,  0.32481298,  1.0023816 , -1.4045309 ,
       -0.01840224, -1.6997966 ,  0.49405205,  1.250316  , -0.33040217,
        0.64891267, -1.7125696 , -2.0854886 ,  1.0396165 , -0.93020767,
       -0.2646402 ,  1.1852943 , -0.72823304, -0.3096057 , -0.17142753,
       -0.23001675, -0.38565218,  0.8793498 ,  0.6329371 ,  0.80

In [18]:
model.wv['going']

array([ 0.52408206,  1.0315932 ,  0.60474604,  0.07806875,  0.47546357,
       -0.62914306,  1.3611653 ,  1.4107957 , -3.5091689 ,  0.43005142,
       -0.5337351 , -1.1269639 ,  0.12905888, -0.7648114 ,  2.063487  ,
       -0.11014901, -0.49379447, -0.10046316,  0.47510803, -1.3979099 ,
        1.1178063 , -0.47334203, -1.0428723 ,  0.05525931,  0.43354887,
       -0.49424225, -1.2429304 , -2.112321  ,  0.8597719 , -0.02377722,
       -0.11065354,  1.0554858 ,  0.31573182, -2.300125  , -0.776858  ,
        0.65452296, -1.1238701 , -1.2418047 ,  1.0082325 , -1.6733485 ,
        1.4183801 ,  0.3797727 ,  1.0606495 ,  0.8282275 ,  0.296509  ,
       -0.37544194, -0.37226346,  0.9472702 ,  0.2941302 , -0.43127245,
       -1.2212996 , -1.1959143 , -1.5319699 , -1.0600247 ,  2.2256405 ,
        1.2578552 ,  0.45063332,  0.9271268 ,  0.9360157 , -1.3551235 ,
        0.30479473,  1.7162601 ,  1.6321949 , -0.08268709, -2.6945758 ,
       -1.0127397 ,  0.22651474,  0.6466314 , -2.3021324 , -1.02

In [19]:
len(model.wv.index_to_key)

113140

In [20]:
sp = model.wv['speak'] # it is the vector representation of speak word

In [21]:
fc = model.wv['force']

In [22]:
cosine_similarity([sp,fc])

array([[1.0000001 , 0.37731397],
       [0.37731397, 0.9999998 ]], dtype=float32)

In [23]:
model.wv.index_to_key

['movie',
 'film',
 'one',
 'like',
 'good',
 'even',
 'would',
 'time',
 'really',
 'story',
 'see',
 'much',
 'well',
 'bad',
 'get',
 'also',
 'great',
 'people',
 'first',
 'dont',
 'films',
 'movies',
 'made',
 'could',
 'way',
 'make',
 'characters',
 'think',
 'watch',
 'seen',
 'character',
 'many',
 'two',
 'never',
 'love',
 'acting',
 'plot',
 'best',
 'know',
 'little',
 'show',
 'life',
 'ever',
 'better',
 'still',
 'scene',
 'say',
 'end',
 'man',
 'scenes',
 'something',
 'go',
 'im',
 'back',
 'real',
 'thing',
 'watching',
 'actors',
 'doesnt',
 'didnt',
 'years',
 'though',
 'funny',
 'actually',
 'another',
 'find',
 'makes',
 'work',
 'look',
 'nothing',
 'going',
 'lot',
 'new',
 'every',
 'old',
 'us',
 'part',
 'cant',
 'director',
 'thats',
 'quite',
 'cast',
 'things',
 'got',
 'pretty',
 'want',
 'enough',
 'around',
 'world',
 'seems',
 'however',
 'young',
 'take',
 'fact',
 'horror',
 'thought',
 'ive',
 'give',
 'big',
 'may',
 'long',
 'always',
 'withou

In [24]:
print("Training reviews:", len(processed_train))
print("Testing reviews:", len(processed_test))
print("Word2Vec vocabulary:", len(model.wv))
print("Vector size:", model.vector_size)

Training reviews: 25000
Testing reviews: 25000
Word2Vec vocabulary: 113140
Vector size: 100


In [25]:
review = processed_train.iloc[0]

# converting each word into its word2vec vector
vectors = [model.wv[word] for word in review if word in model.wv]
print("Number of words:",len(review))
print("Number of vectors:",len(vectors))
print("Shape of one word vector:",vectors[0].shape)

Number of words: 61
Number of vectors: 61
Shape of one word vector: (100,)


In [26]:
len(processed_train.iloc[1])

142

In [27]:
# Checking the number of words in each training review
review_lengths = processed_train.apply(len)

print("Shortest review:", review_lengths.min())
print("Longest review:", review_lengths.max())
print("Average review length:", review_lengths.mean())
print("95th percentile:", review_lengths.quantile(0.95))

Shortest review: 3
Longest review: 1437
Average review length: 120.00644
95th percentile: 309.0


In [28]:
# converting each review into a sequnce of word2vec vectors
X_train_vectors = [[model.wv[word] for word in review if word in model.wv]
for review in processed_train]



In [29]:
X_test_vectors = [
    [model.wv[word] for word in review if word in model.wv]
    for review in processed_test
]

In [30]:
print("Training reviews:", len(X_train_vectors))
print("Testing reviews:", len(X_test_vectors))

print("First training review — words:", len(X_train_vectors[0]))
print("First training review — vector shape:", X_train_vectors[0][0].shape)

Training reviews: 25000
Testing reviews: 25000
First training review — words: 61
First training review — vector shape: (100,)


In [31]:
# padding and truncation

import numpy as np

max_length = 309
vector_size = model.vector_size

X_train_padded = np.zeros(
    (len(X_train_vectors), max_length, vector_size),
    dtype=np.float32
)

for i, review_vectors in enumerate(X_train_vectors):
    review_vectors = np.asarray(review_vectors, dtype=np.float32)
    length = min(len(review_vectors), max_length)
    X_train_padded[i, :length] = review_vectors[:length]

    

In [32]:
print("Training array shape:", X_train_padded.shape)
print("Training array data type:", X_train_padded.dtype)

Training array shape: (25000, 309, 100)
Training array data type: float32


In [33]:
from sklearn.preprocessing import LabelEncoder

In [34]:
le = LabelEncoder()

y_train_encoded = le.fit_transform(Y_train)
y_test_encoded = le.transform(Y_test)

print("Classes:", le.classes_)
print("Training labels shape:", y_train_encoded.shape)
print("First 10 labels:", y_train_encoded[:10])

Classes: ['negative' 'positive']
Training labels shape: (25000,)
First 10 labels: [1 0 1 1 1 0 1 1 0 1]


In [54]:

print("Original review:")
print(X_test.iloc[35])

print("\nProcessed tokens:")
print(processed_test.iloc[35])

print("\nNumber of processed tokens:",
      len(processed_test.iloc[35]))

print("\nNumber of vectors:",
      len(X_test_vectors[35]))

Original review:
Shame Shame Shame on UA/DW for what you do! <br /><br />I was appalled. <br /><br />Do NOT take kids to see this movie. The humor is totally inappropriate for children - plus they'll be bored and disappointed. Certainly *we all* have read Theo's wonderful children book and certainly we have expectations...but this is pure trash. Dr. Seuss would be ashamed and certainly would've never given his "thumbs up" at such a dastardly attempt to capitalize on a classic.<br /><br />What a pity. <br /><br />Spend your money on the book. If you own a copy, then buy the book and donate it to a Toys for Tots program. This movie is NOT worth a "free" ticket viewing.<br /><br />Stick with the book. The tv cartoon version works well if you want a visual portrayal - save your money...seriously. SAVE your money - it will be on cable by saint patty's day.<br /><br />Shame shame shame on what they do!!

Processed tokens:
['shame', 'shame', 'shame', 'uadw', 'appalled', 'take', 'kids', 'see',

In [35]:
import tensorflow as tf

In [36]:
print("TensorFlow version:", tf.__version__)
print("Word2Vec vocabulary size:", len(model.wv))
print("Word2Vec vector size:", model.vector_size)
print("Maximum sequence length:", max_length)

TensorFlow version: 2.21.0
Word2Vec vocabulary size: 113140
Word2Vec vector size: 100
Maximum sequence length: 309


In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input,LSTM,Dense,Dropout

In [38]:
model_lstm = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(309,100)),
    tf.keras.layers.LSTM(64),
    tf.keras.layers.Dense(32,activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

In [39]:
model_lstm.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,353 (173.25 KB)

 Trainable params: 44,353 (173.25 KB)

 Non-trainable params: 0 (0.00 B)

In [40]:
model_lstm.compile(loss='binary_crossentropy',
                   optimizer='adam',
                   metrics=['accuracy'])

In [41]:
print("X_train_padded shape:", X_train_padded.shape)
print("X_train_padded dtype:", X_train_padded.dtype)

print("y_train_encoded shape:", y_train_encoded.shape)
print("y_train_encoded dtype:", y_train_encoded.dtype)

print("X_train_padded memory (GB):",
      X_train_padded.nbytes / (1024**3))

print("Label classes:", np.unique(y_train_encoded))

X_train_padded shape: (25000, 309, 100)
X_train_padded dtype: float32
y_train_encoded shape: (25000,)
y_train_encoded dtype: int64
X_train_padded memory (GB): 2.8777867555618286
Label classes: [0 1]


In [42]:
model_lstm.fit(X_train_padded,y_train_encoded,epochs=10,batch_size=16,validation_split=0.1,verbose=1)

Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 84s 50ms/step - accuracy: 0.5111 - loss: 0.6904 - val_accuracy: 0.5064 - val_loss: 0.6926
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 67s 47ms/step - accuracy: 0.6092 - loss: 0.6427 - val_accuracy: 0.7124 - val_loss: 0.5884
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 47ms/step - accuracy: 0.7413 - loss: 0.5738 - val_accuracy: 0.7668 - val_loss: 0.5429
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 47ms/step - accuracy: 0.7160 - loss: 0.5798 - val_accuracy: 0.6988 - val_loss: 0.5928
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 47ms/step - accuracy: 0.5959 - loss: 0.6367 - val_accuracy: 0.5064 - val_loss: 0.6902
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 47ms/step - accuracy: 0.5148 - loss: 0.6855 - val_accuracy: 0.5172 - val_loss: 0.6827
Epoch 7/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 47ms/step - accuracy: 0.6816 - loss: 0.5438 - val_accuracy: 0.8372 - val_loss: 0.3803
Epoch 8/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 47ms/step - accuracy: 0.8568 -

In [46]:
X_test_padded = np.zeros(
    (len(X_test_vectors), max_length, vector_size),
    dtype=np.float32
)

for i, review_vectors in enumerate(X_test_vectors):
    length = min(len(review_vectors), max_length)

    if length > 0:
        X_test_padded[i, :length] = np.asarray(
            review_vectors[:length],
            dtype=np.float32
        )

print(X_test_padded.shape)

(25000, 309, 100)


In [48]:
y_pred_prob = model_lstm.predict(X_test_padded)

y_pred = (y_pred_prob >= 0.5).astype(int)

print("Predictions shape:", y_pred.shape)
print("First 10 predictions:", y_pred[:10].flatten())
print("First 10 actual labels:", y_test_encoded[:10])

782/782 ━━━━━━━━━━━━━━━━━━━━ 70s 65ms/step
Predictions shape: (25000, 1)
First 10 predictions: [0 0 0 0 0 1 0 1 0 0]
First 10 actual labels: [0 0 1 0 0 1 0 1 0 0]


In [49]:
from sklearn.metrics import accuracy_score

test_accuracy = accuracy_score(y_test_encoded,y_pred)

print("Test Accuracy:",test_accuracy)
print("Test Accuracy (%):", test_accuracy * 100)

Test Accuracy: 0.85384
Test Accuracy (%): 85.384


In [50]:
# confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_encoded, y_pred)

print(cm)

[[11765   735]
 [ 2919  9581]]


In [55]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test_encoded,
    y_pred,
    target_names=['Negative','Positive']
)
)

              precision    recall  f1-score   support

    Negative       0.80      0.94      0.87     12500
    Positive       0.93      0.77      0.84     12500

    accuracy                           0.85     25000
   macro avg       0.86      0.85      0.85     25000
weighted avg       0.86      0.85      0.85     25000



In [56]:
model_lstm.save("model_lstm.keras")
model.save("word2vec.model")

print("models saved successfully.")

models saved successfully.
